<a href="https://colab.research.google.com/github/Ezgicode/FlyRAnk-AI/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ezgicode/FlyRAnk-AI/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

The feature vector is built from observable search signals measured only from 1–15 March 2026. The later 16–31 March impression outcome is used only to create the decline proxy and is never included in the final model features. Pseudonymized client and content IDs are used for grouping and reproducibility only, never as model inputs.

In [ ]:
!pip -q install duckdb

import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("Add your Hugging Face read token as the Colab Secret named HF_TOKEN.")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
MARCH = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

feature_sql = f"""
WITH march_page_days AS (
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,
        gsc_impressions,
        gsc_clicks,
        gsc_sum_position
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
      AND report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
),
page_windows AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(CASE WHEN report_date <= DATE '2026-03-15'
                 THEN gsc_impressions ELSE 0 END) AS prior_15d_impressions,

        SUM(CASE WHEN report_date <= DATE '2026-03-15'
                 THEN gsc_clicks ELSE 0 END) AS prior_15d_clicks,

        SUM(CASE WHEN report_date <= DATE '2026-03-15'
                 THEN gsc_sum_position ELSE 0 END)
        / NULLIF(
            SUM(CASE WHEN report_date <= DATE '2026-03-15'
                     THEN gsc_impressions ELSE 0 END),
            0
        ) AS prior_15d_avg_position,

        COUNT(DISTINCT CASE WHEN report_date <= DATE '2026-03-15'
                            THEN report_date END) AS prior_15d_active_days,

        SUM(CASE WHEN report_date >= DATE '2026-03-16'
                 THEN gsc_impressions ELSE 0 END) AS next_16d_impressions
    FROM march_page_days
    GROUP BY client_hash_id, content_hash_id
)
SELECT *
FROM page_windows
WHERE prior_15d_impressions >= 100
"""

feature_frame = con.sql(feature_sql).df()

feature_frame["prior_15d_ctr"] = (
    feature_frame["prior_15d_clicks"]
    / feature_frame["prior_15d_impressions"]
)

feature_frame["is_next_15d_declining"] = (
    feature_frame["next_16d_impressions"]
    <= 0.80 * feature_frame["prior_15d_impressions"]
).astype(int)

final_feature_columns = [
    "prior_15d_impressions",
    "prior_15d_clicks",
    "prior_15d_avg_position",
    "prior_15d_ctr",
    "prior_15d_active_days",
]

X = feature_frame[final_feature_columns].copy()

# Engineering is based only on the prior feature window.
X["log_prior_15d_impressions"] = np.log1p(X["prior_15d_impressions"])
X["log_prior_15d_clicks"] = np.log1p(X["prior_15d_clicks"])

# All final features are numeric; no categorical feature or ID is used.
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))

y = feature_frame["is_next_15d_declining"].copy()
groups = feature_frame["client_hash_id"].copy()

print(f"Feature-frame rows: {len(feature_frame):,}")
print(f"Final feature count: {X.shape[1]}")
print(f"Later-window decline proxy rate: {y.mean():.1%}")

display(X.head())

Feature-frame rows: 77,540
Final feature count: 7
Later-window decline proxy rate: 28.6%


,prior_15d_impressions,prior_15d_clicks,prior_15d_avg_position,prior_15d_ctr,prior_15d_active_days,log_prior_15d_impressions,log_prior_15d_clicks
0,429.0,2.0,4.386946,0.004662,15,6.063785,1.098612
1,628.0,1.0,5.265924,0.001592,15,6.444131,0.693147
2,1280.0,9.0,4.144531,0.007031,15,7.155396,2.302585
3,2531.0,10.0,4.861320,0.003951,15,7.836765,2.397895
4,243.0,1.0,14.534979,0.004115,15,5.497168,0.693147


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

All final features are numeric and measured before the prediction boundary. Missing numeric values are replaced with the feature median after impossible infinite values are converted to missing values. No categorical variables are used in this final feature vector; position bands may be used later for human-readable reason codes, but the raw prior average position is the model input.

In [ ]:
feature_notes = pd.DataFrame(
    [
        {
            "feature": "prior_15d_impressions",
            "meaning": "Observed search impressions during 1–15 March 2026.",
            "missing_handling": "Median after cleaning; eligibility requires at least 100 impressions.",
            "available_before_outcome": "Yes",
        },
        {
            "feature": "prior_15d_clicks",
            "meaning": "Observed search clicks during 1–15 March 2026.",
            "missing_handling": "Median after cleaning.",
            "available_before_outcome": "Yes",
        },
        {
            "feature": "prior_15d_avg_position",
            "meaning": "Impression-weighted observed average search position during 1–15 March.",
            "missing_handling": "Median after cleaning.",
            "available_before_outcome": "Yes",
        },
        {
            "feature": "prior_15d_ctr",
            "meaning": "Prior clicks divided by prior impressions.",
            "missing_handling": "Median after cleaning; prior impressions are at least 100.",
            "available_before_outcome": "Yes",
        },
        {
            "feature": "prior_15d_active_days",
            "meaning": "Number of observed search-data days in the prior window.",
            "missing_handling": "Median after cleaning.",
            "available_before_outcome": "Yes",
        },
        {
            "feature": "log_prior_15d_impressions / log_prior_15d_clicks",
            "meaning": "Log transforms that reduce the influence of heavy-tailed volume fields.",
            "missing_handling": "Built after positive-count fields; cleaned numeric median fill.",
            "available_before_outcome": "Yes",
        },
    ]
)

display(feature_notes)

print("Missing values in final vector:")
display(X.isna().sum().to_frame("missing_values"))

assert X.isna().sum().sum() == 0
assert not {"client_hash_id", "content_hash_id"}.intersection(X.columns)

,feature,meaning,missing_handling,available_before_outcome
0,prior_15d_impressions,Observed search impressions during 1–15 March ...,Median after cleaning; eligibility requires at...,Yes
1,prior_15d_clicks,Observed search clicks during 1–15 March 2026.,Median after cleaning.,Yes
2,prior_15d_avg_position,Impression-weighted observed average search po...,Median after cleaning.,Yes
3,prior_15d_ctr,Prior clicks divided by prior impressions.,Median after cleaning; prior impressions are a...,Yes
4,prior_15d_active_days,Number of observed search-data days in the pri...,Median after cleaning.,Yes
5,log_prior_15d_impressions / log_prior_15d_clicks,Log transforms that reduce the influence of he...,Built after positive-count fields; cleaned num...,Yes


Missing values in final vector:


,missing_values
prior_15d_impressions,0
prior_15d_clicks,0
prior_15d_avg_position,0
prior_15d_ctr,0
prior_15d_active_days,0
log_prior_15d_impressions,0
log_prior_15d_clicks,0


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

The leakage hunt rejects future-window fields, label-derived fields, FlyRank product decision fields, and pseudonymized IDs. The direct test below demonstrates why the later-window impression field cannot be included: it can reproduce the decline proxy by definition and would therefore make evaluation misleading.

In [ ]:
forbidden_patterns = [
    "next_",
    "is_next_",
    "trend_",
    "label",
    "target",
    "health",
    "priority",
    "action",
    "refresh",
    "client_hash_id",
    "content_hash_id",
    "report_date",
]

violations = [
    column
    for column in X.columns
    if any(pattern in column.lower() for pattern in forbidden_patterns)
]

print("Forbidden-pattern matches in final model feature vector:", violations)
assert len(violations) == 0

# Demonstration only: this is intentionally leaky and is NOT added to X.
leaky_decline_rule = (
    feature_frame["next_16d_impressions"]
    <= 0.80 * feature_frame["prior_15d_impressions"]
).astype(int)

leaky_agreement = (leaky_decline_rule == y).mean()

leakage_check = pd.DataFrame(
    [
        {
            "test": "Future-window impressions recreate the label",
            "result": f"{leaky_agreement:.1%} agreement with the decline proxy",
            "verdict": "LEAKY — excluded from final features",
        },
        {
            "test": "Pseudonymized IDs in final feature vector",
            "result": "No IDs present in X",
            "verdict": "PASS",
        },
        {
            "test": "Product flags or label-derived fields in X",
            "result": "No forbidden-pattern matches",
            "verdict": "PASS",
        },
    ]
)

display(leakage_check)

assert leaky_agreement == 1.0

Forbidden-pattern matches in final model feature vector: []


,test,result,verdict
0,Future-window impressions recreate the label,100.0% agreement with the decline proxy,LEAKY — excluded from final features
1,Pseudonymized IDs in final feature vector,No IDs present in X,PASS
2,Product flags or label-derived fields in X,No forbidden-pattern matches,PASS


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

I excluded fields that would reveal the answer, reproduce an existing product decision, identify a page or client, or use information unavailable at the 15 March decision point. These exclusions make the result an honest, directional decision-support score rather than a circular copy of future outcomes or existing rules.

In [ ]:
excluded_fields = pd.DataFrame(
    [
        {
            "excluded_field_or_group": "next_16d_impressions",
            "why_excluded": "It belongs to the later outcome window and directly defines the decline proxy.",
        },
        {
            "excluded_field_or_group": "is_next_15d_declining",
            "why_excluded": "This is the target label, not a feature.",
        },
        {
            "excluded_field_or_group": "trend_direction and trend_pct",
            "why_excluded": "Label-derived fields would make the analysis circular.",
        },
        {
            "excluded_field_or_group": "health_score, priority_score, action labels, refresh flags",
            "why_excluded": "These are existing product decisions rather than raw observable signals.",
        },
        {
            "excluded_field_or_group": "client_hash_id and content_hash_id",
            "why_excluded": "Used only for grouping and reproducibility; IDs should not become predictive signals.",
        },
        {
            "excluded_field_or_group": "Raw URLs, client names, and private queries",
            "why_excluded": "Not required for the task and must never appear in public work.",
        },
    ]
)

display(excluded_fields)

print("Final model feature columns:")
print(list(X.columns))

,excluded_field_or_group,why_excluded
0,next_16d_impressions,It belongs to the later outcome window and dir...
1,is_next_15d_declining,"This is the target label, not a feature."
2,trend_direction and trend_pct,Label-derived fields would make the analysis c...
3,"health_score, priority_score, action labels, r...",These are existing product decisions rather th...
4,client_hash_id and content_hash_id,Used only for grouping and reproducibility; ID...
5,"Raw URLs, client names, and private queries",Not required for the task and must never appea...


Final model feature columns:
['prior_15d_impressions', 'prior_15d_clicks', 'prior_15d_avg_position', 'prior_15d_ctr', 'prior_15d_active_days', 'log_prior_15d_impressions', 'log_prior_15d_clicks']


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.